In [1]:
import numpy as np 
import pandas as pd

In [2]:
df=pd.read_csv("hotels.csv")

In [3]:
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='object')

## Brief-number checks with Pandas

The cells below compare the brief with the bookings file. Cancellation outcomes are calculated only for train and choose. The proof row shows dates and booking count, but its cancellation labels are held for the final proof run. Costs and walk limits are client assumptions, not measurements from this CSV.

The charter's booking-level decision-unit guess is intentionally marked as probably wrong; the scoring contract's action unit is one hotel on one arrival night.

### 1. Load the bookings and build the arrival date

In [4]:
from pathlib import Path
import pandas as pd

# Find the CSV when running from notebooks/, data/, or the repository root.
csv_path = Path("hotels.csv")
if not csv_path.exists():
    csv_path = Path("../data/hotels.csv")
if not csv_path.exists():
    csv_path = Path("data/hotels.csv")

bookings = pd.read_csv(csv_path)
row_count, column_count = bookings.shape

# Combine the arrival date fields so the brief's time periods are easy to check.
date_text = (
    bookings["arrival_date_year"].astype(str)
    + "-"
    + bookings["arrival_date_month"]
    + "-"
    + bookings["arrival_date_day_of_month"].astype(str)
)
bookings["arrival_date"] = pd.to_datetime(date_text, format="%Y-%B-%d")


### 2. Check the train, choose, and proof periods

In [5]:
train = bookings[bookings["arrival_date"] < "2016-07-01"]
choose = bookings[bookings["arrival_date"].between("2016-07-01", "2016-12-31")]
proof = bookings[bookings["arrival_date"].between("2017-01-01", "2017-08-31")]

# Keep proof-period cancellation outcomes out of this early data check.
period_summary = pd.DataFrame([
    {
        "Period": "Train",
        "Arrival dates": f"{train['arrival_date'].min():%Y-%m-%d} to {train['arrival_date'].max():%Y-%m-%d}",
        "Bookings": len(train),
        "Cancellation rate": f"{train['is_canceled'].mean():.1%}",
    },
    {
        "Period": "Choose",
        "Arrival dates": f"{choose['arrival_date'].min():%Y-%m-%d} to {choose['arrival_date'].max():%Y-%m-%d}",
        "Bookings": len(choose),
        "Cancellation rate": f"{choose['is_canceled'].mean():.1%}",
    },
    {
        "Period": "Proof",
        "Arrival dates": f"{proof['arrival_date'].min():%Y-%m-%d} to {proof['arrival_date'].max():%Y-%m-%d}",
        "Bookings": len(proof),
        "Cancellation rate": "Held for final proof run",
    },
])
period_summary


Period,Arrival dates,Bookings,Cancellation rate
Train,2015-07-01 to 2016-06-30,49157,35.8%
Choose,2016-07-01 to 2016-12-31,29546,36.9%
Proof,2017-01-01 to 2017-08-31,40687,Held for final proof run


### 3. Compare the cancellation claim without using proof outcomes

In [6]:
# Compare the brief's rough 25% claim using train and choose only.
pre_proof = pd.concat([train, choose])
overall_rate = pre_proof["is_canceled"].mean()
hotel_rates = pre_proof.groupby("hotel")["is_canceled"].mean()

cancellation_summary = pd.DataFrame([
    {"Group": "Train + choose", "Cancellation rate": f"{overall_rate:.1%}"},
    *[
        {"Group": hotel, "Cancellation rate": f"{rate:.1%}"}
        for hotel, rate in hotel_rates.items()
    ],
])
cancellation_summary


Group,Cancellation rate
Train + choose,36.2%
City Hotel,41.3%
Resort Hotel,26.3%


### 4. List the remaining client numbers and claims

In [7]:
# These are client-provided settings or claims, not all measurable from the CSV.
brief_checks = pd.DataFrame([
    {
        "Brief number": "Two hotels",
        "Data result": f"{bookings['hotel'].nunique()} distinct hotels",
        "Note": "Matches the brief" if bookings["hotel"].nunique() == 2 else "Check the source file",
    },
    {
        "Brief number": "119,390 bookings; 32 columns",
        "Data result": f"{row_count:,} rows; {column_count} columns",
        "Note": "Matches the brief" if (row_count, column_count) == (119390, 32) else "Check the source file",
    },
    {
        "Brief number": "Arrivals: July 2015–August 2017",
        "Data result": f"{bookings['arrival_date'].min():%Y-%m-%d} to {bookings['arrival_date'].max():%Y-%m-%d}",
        "Note": "Compare with the stated range",
    },
    {
        "Brief number": "About 1 in 4 cancellations (~25%)",
        "Data result": f"{overall_rate:.1%} on train + choose",
        "Note": "Higher than the rough claim; proof outcomes excluded",
    },
    {
        "Brief number": "1 walk per 100 / 10 per 1,000",
        "Data result": "10 per 1,000 = 1.0%",
        "Note": "Same stated ceiling; not an observed walk rate",
    },
    {
        "Brief number": "EUR 300 per walked guest",
        "Data result": "Client-stated cost",
        "Note": "Not measurable from booking rows",
    },
    {
        "Brief number": "Current rule has ‘a few rooms’ for safety",
        "Data result": "No exact margin in the file",
        "Note": "Ask the revenue manager",
    },
    {
        "Brief number": "Next 14 nights",
        "Data result": "14-night output requirement",
        "Note": "MVP has not been run",
    },
    {
        "Brief number": "Vendor demo: 95% accuracy",
        "Data result": "No demo results supplied",
        "Note": "Unverified claim; use business cost measures",
    },
])
brief_checks


Brief number,Data result,Note
Two hotels,2 distinct hotels,Matches the brief
"119,390 bookings; 32 columns","119,390 rows; 32 columns",Matches the brief
Arrivals: July 2015–August 2017,2015-07-01 to 2017-08-31,Compare with the stated range
About 1 in 4 cancellations (~25%),36.2% on train + choose,Higher than the rough claim; proof outcomes excluded
"1 walk per 100 / 10 per 1,000","10 per 1,000 = 1.0%",Same stated ceiling; not an observed walk rate
EUR 300 per walked guest,Client-stated cost,Not measurable from booking rows
Current rule has ‘a few rooms’ for safety,No exact margin in the file,Ask the revenue manager
Next 14 nights,14-night output requirement,MVP has not been run
Vendor demo: 95% accuracy,No demo results supplied,Unverified claim; use business cost measures
